# EXP20: 最优参数组合验证

汇总 EXP01-26 各维度冠军参数，组合测试 + K-Fold 验证。

## 冠军参数来源
| 维度 | 冠军值 | 来源 | Sharpe |
|---|---|---|---|
| KC EMA | 25 | EXP13 | 1.84 |
| KC Multiplier | 1.2 | EXP13 | 1.84 |
| EMA Trend | 150 | EXP14 | 1.65 |
| SL ATR | 5.0 | EXP04 | 2.39 |
| Trail Act/Dist | 0.6/0.2 | EXP04 | 2.39 |
| ADX | 18 | EXP09 | 已是当前值 |
| TP | 6.0 | EXP11 | 1.57 |
| KC max_hold | 60 (不限制) | EXP01 | 1.55 |
| Choppy/KC_only | 0.35/0.60 | EXP15 | 保持不变(0.65是悬崖) |

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import config
import pandas as pd
import numpy as np

print('Loading default data (KC_EMA20, KC_mult1.5, EMA100)...')
data_default = DataBundle.load_default()

print('Loading custom data (KC_EMA25, KC_mult1.2, EMA150)...')
data_custom = DataBundle.load_custom(kc_ema=25, kc_mult=1.2, ema_trend=150)

print('Data loaded')

## Step 1: Baseline vs 各维度独立冠军

In [ ]:
# =========================================
# 当前实盘 Baseline
# =========================================
BASELINE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data_default, "A_Baseline", **BASELINE_KWARGS)
print(f"A Baseline:  Sharpe={baseline['sharpe']:.2f}, PnL=${baseline['total_pnl']:.0f}, N={baseline['n']}, MaxDD=${baseline.get('max_dd',0):.0f}")

In [ ]:
# =========================================
# B: 仅更换 KC 通道参数 (EXP13)
# =========================================
b_kc = run_variant(data_custom, "B_KC25_M1.2_EMA150", **BASELINE_KWARGS)
print(f"B KC25/M1.2/EMA150: Sharpe={b_kc['sharpe']:.2f}, PnL=${b_kc['total_pnl']:.0f}, N={b_kc['n']}")

# =========================================
# C: 仅更换 SL/Trail (EXP04)
# =========================================
C_KWARGS = {
    **BASELINE_KWARGS,
    "sl_atr_mult": 5.0,
    "trailing_activate_atr": 0.6,
    "trailing_distance_atr": 0.2,
}
c_sl_trail = run_variant(data_default, "C_SL5.0_T0.6/0.2", **C_KWARGS)
print(f"C SL5.0/T0.6/0.2:   Sharpe={c_sl_trail['sharpe']:.2f}, PnL=${c_sl_trail['total_pnl']:.0f}, N={c_sl_trail['n']}")

# =========================================
# D: 仅更换 TP (EXP11)
# =========================================
d_tp = run_variant(data_default, "D_TP6.0", **{**BASELINE_KWARGS, "tp_atr_mult": 6.0})
print(f"D TP6.0:             Sharpe={d_tp['sharpe']:.2f}, PnL=${d_tp['total_pnl']:.0f}, N={d_tp['n']}")

## Step 2: 组合冠军

In [ ]:
# =========================================
# E: KC通道 + SL/Trail 组合
# =========================================
E_KWARGS = {
    **BASELINE_KWARGS,
    "sl_atr_mult": 5.0,
    "trailing_activate_atr": 0.6,
    "trailing_distance_atr": 0.2,
}
e_combo2 = run_variant(data_custom, "E_KC+SL/Trail", **E_KWARGS)
print(f"E KC+SL/Trail:   Sharpe={e_combo2['sharpe']:.2f}, PnL=${e_combo2['total_pnl']:.0f}, N={e_combo2['n']}")

# =========================================
# F: 全部冠军组合 (KC + SL/Trail + TP)
# =========================================
F_KWARGS = {
    **BASELINE_KWARGS,
    "sl_atr_mult": 5.0,
    "tp_atr_mult": 6.0,
    "trailing_activate_atr": 0.6,
    "trailing_distance_atr": 0.2,
}
f_full = run_variant(data_custom, "F_FullCombo", **F_KWARGS)
print(f"F Full Combo:    Sharpe={f_full['sharpe']:.2f}, PnL=${f_full['total_pnl']:.0f}, N={f_full['n']}")

In [ ]:
# Summary table
all_results = [
    ('A Baseline', baseline),
    ('B KC25/M1.2/EMA150', b_kc),
    ('C SL5.0/T0.6/0.2', c_sl_trail),
    ('D TP6.0', d_tp),
    ('E KC+SL/Trail', e_combo2),
    ('F Full Combo', f_full),
]

print(f"\n{'Config':25s} {'Sharpe':>7s} {'PnL':>10s} {'N':>6s} {'MaxDD':>8s} {'$/trade':>8s}")
print('-' * 70)
for name, r in all_results:
    per = r['total_pnl'] / r['n'] if r['n'] > 0 else 0
    print(f"{name:25s} {r['sharpe']:7.2f} ${r['total_pnl']:9.0f} {r['n']:6d} ${r.get('max_dd',0):7.0f} ${per:7.2f}")

print(f"\nBaseline -> Full Combo: Sharpe {f_full['sharpe']-baseline['sharpe']:+.2f}, PnL ${f_full['total_pnl']-baseline['total_pnl']:+.0f}")

## Step 3: K-Fold 交叉验证

In [ ]:
print("\n=== K-Fold: Baseline vs Full Combo ===")
base_folds = run_kfold(data_default, BASELINE_KWARGS, label_prefix="Base_")
combo_folds = run_kfold(data_custom, F_KWARGS, label_prefix="Combo_")

print(f"\n{'Fold':15s} {'Base Sharpe':>12s} {'Combo Sharpe':>13s} {'Delta':>8s} {'Winner':>8s}")
print('-' * 60)
for b, c in zip(base_folds, combo_folds):
    d = c['sharpe'] - b['sharpe']
    w = 'Combo' if d > 0 else 'Base'
    print(f"{b['fold']:15s} {b['sharpe']:12.2f} {c['sharpe']:13.2f} {d:+8.2f} {w:>8s}")

wins = sum(1 for b, c in zip(base_folds, combo_folds) if c['sharpe'] > b['sharpe'])
avg_base = np.mean([b['sharpe'] for b in base_folds])
avg_combo = np.mean([c['sharpe'] for c in combo_folds])
print(f"\nAvg Sharpe: Base={avg_base:.2f}, Combo={avg_combo:.2f}, Delta={avg_combo-avg_base:+.2f}")
print(f"Combo wins {wins}/{len(base_folds)} folds")

## Step 4: 保守组合 (仅 KC 通道优化，不改 SL/Trail)

In [ ]:
print("\n=== K-Fold: Baseline vs KC-Only Upgrade ===")
kc_folds = run_kfold(data_custom, BASELINE_KWARGS, label_prefix="KC_")

print(f"\n{'Fold':15s} {'Base':>8s} {'KC25/1.2':>10s} {'Combo':>8s}")
print('-' * 45)
for b, k, c in zip(base_folds, kc_folds, combo_folds):
    print(f"{b['fold']:15s} {b['sharpe']:8.2f} {k['sharpe']:10.2f} {c['sharpe']:8.2f}")

kc_wins = sum(1 for b, k in zip(base_folds, kc_folds) if k['sharpe'] > b['sharpe'])
avg_kc = np.mean([k['sharpe'] for k in kc_folds])
print(f"\nKC-only wins {kc_wins}/{len(base_folds)} folds, Avg Sharpe={avg_kc:.2f} (Base={avg_base:.2f})")
print(f"\nRecommendation:")
print(f"  KC-only upgrade safe? {'YES' if kc_wins >= 4 else 'CAUTION'}")
print(f"  Full combo safe? {'YES' if wins >= 4 else 'CAUTION'}")

In [ ]:
import json
summary = {
    'baseline': {'sharpe': baseline['sharpe'], 'pnl': baseline['total_pnl'], 'n': baseline['n']},
    'kc_only': {'sharpe': b_kc['sharpe'], 'pnl': b_kc['total_pnl'], 'n': b_kc['n']},
    'sl_trail': {'sharpe': c_sl_trail['sharpe'], 'pnl': c_sl_trail['total_pnl'], 'n': c_sl_trail['n']},
    'full_combo': {'sharpe': f_full['sharpe'], 'pnl': f_full['total_pnl'], 'n': f_full['n']},
    'kfold_combo_wins': wins,
    'kfold_kc_wins': kc_wins,
    'avg_sharpe': {'base': avg_base, 'kc': avg_kc, 'combo': avg_combo},
    'champion_params': {
        'kc_ema': 25, 'kc_mult': 1.2, 'ema_trend': 150,
        'sl_atr_mult': 5.0, 'tp_atr_mult': 6.0,
        'trailing_activate_atr': 0.6, 'trailing_distance_atr': 0.2,
        'choppy': 0.35, 'kc_only': 0.60,
    }
}
with open('../data/exp20_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved')
print(json.dumps(summary, indent=2))